# 08 · Domain-Adversarial Strength Sweep

Domain-adversarial training is sensitive to the adversarial weight (lambda). This
notebook trains configuration E at several peak lambda values on the Mendeley-held-out
fold to identify the setting that best balances grading accuracy and domain invariance.
The selected value is used for the leave-one-dataset-out folds.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
from pathlib import Path
import torch
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
if not torch.cuda.is_available(): raise RuntimeError('No GPU.')
print('GPU:', torch.cuda.get_device_name(0))
manifest=T.prepare_local_data()

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Copied array in 30s
Loaded array (61558, 224, 224) in 1s


In [2]:
def make_splits(manifest, test_dataset, seed=0):
    pool=manifest[manifest['dataset']!=test_dataset].copy()
    test=manifest[manifest['dataset']==test_dataset].copy()
    if 'split' in pool.columns and pool['split'].isin(['train','val']).any():
        tr=pool[pool['split'].isin(['train','TRAIN'])]; va=pool[pool['split'].isin(['val','VAL','validation'])]
        if len(va)==0: va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    else:
        va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    return tr.reset_index(drop=True), va.reset_index(drop=True), test.reset_index(drop=True)

## Lambda sweep

In [3]:
test_ds=config.LODO_FOLDS[config.ABLATION_FOLD]
results=[]
for lam in config.GRL_LAMBDA_SWEEP:
    mech=dict(config.FULL_METHOD); mech['grl_lambda_max']=lam
    run=f'grlsweep_lambda{lam}_seed0'
    tr,va,te=make_splits(manifest,test_ds,seed=0)
    print(f'--- {run} (lambda={lam}) ---',flush=True)
    r=T.run_training(run,tr,va,te,mech,0,config.CKPT_DIR,config.RESULTS_DIR,log_fn=print)
    r['grl_lambda_max']=lam; results.append(r)
df=pd.DataFrame(results)
print(df[['run_name','internal_acc5','external_acc5','external_qwk','gap']].to_string(index=False))
df.to_csv(str(config.RESULTS_DIR/'grl_lambda_sweep.csv'),index=False)
best=df.loc[df['external_acc5'].idxmax()]
print(f'\nBest lambda: {best["grl_lambda_max"]} (external {best["external_acc5"]:.3f})')
print('Set config.GRL_LAMBDA_MAX to this value before running LODO folds 04-07.')

--- grlsweep_lambda0.1_seed0 (lambda=0.1) ---
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 222MB/s]


  [grlsweep_lambda0.1_seed0] ep1/18 loss=3.174 tr=0.462 val=0.413 gap=+0.049 qwk=0.235 grl=0.00 (126s)
  [grlsweep_lambda0.1_seed0] ep2/18 loss=2.637 tr=0.514 val=0.466 gap=+0.048 qwk=0.450 grl=0.03 (106s)
  [grlsweep_lambda0.1_seed0] ep3/18 loss=2.402 tr=0.545 val=0.467 gap=+0.078 qwk=0.504 grl=0.05 (106s)
  [grlsweep_lambda0.1_seed0] ep4/18 loss=2.305 tr=0.556 val=0.504 gap=+0.052 qwk=0.545 grl=0.07 (106s)
  [grlsweep_lambda0.1_seed0] ep5/18 loss=2.235 tr=0.570 val=0.494 gap=+0.076 qwk=0.537 grl=0.08 (107s)
  [grlsweep_lambda0.1_seed0] ep6/18 loss=2.125 tr=0.579 val=0.506 gap=+0.073 qwk=0.547 grl=0.09 (107s)
  [grlsweep_lambda0.1_seed0] ep7/18 loss=2.035 tr=0.584 val=0.513 gap=+0.071 qwk=0.553 grl=0.09 (107s)
  [grlsweep_lambda0.1_seed0] ep8/18 loss=1.963 tr=0.582 val=0.526 gap=+0.056 qwk=0.583 grl=0.10 (107s)
  [grlsweep_lambda0.1_seed0] ep9/18 loss=1.900 tr=0.582 val=0.529 gap=+0.053 qwk=0.573 grl=0.10 (106s)
  [grlsweep_lambda0.1_seed0] ep10/18 loss=1.824 tr=0.586 val=0.528 gap=+0